# SDSS spektar → CSV

Ovaj notebook učitava SDSS spektralni FITS fajl i generiše CSV sa kolonama:
- `Talasna_duzina_A` — talasna dužina u angstremima (Å)
- `Relativni_fluks` — fluks normalizovan na medijanu (medijana = 100)

U zaglavlju CSV-a se čuvaju najvažniji podaci o posmatranju.

## 1. Instalacija potrebnih biblioteka

In [ ]:
!pip install astropy -q

## 2. Učitavanje FITS fajla

Izaberi jedan od dva načina:
- **2a** — ručni upload fajla sa računara
- **2b** — montiranje Google Drive-a (ako je fajl tamo)

In [ ]:
# --- 2a: Ručni upload ---
from google.colab import files

uploaded = files.upload()  # otvori dijalog za izbor fajla
putanja_fits = list(uploaded.keys())[0]
print(f"Učitan fajl: {putanja_fits}")

In [ ]:
# --- 2b: Google Drive (alternativa) ---
# from google.colab import drive
# drive.mount('/content/drive')
# putanja_fits = "/content/drive/MyDrive/spec-3249-54880-0090.fits"  # prilagodi putanju

## 3. Obrada spektra

In [ ]:
import os
import numpy as np
from astropy.io import fits

VAZNI_KLJUCEVI = [
    ("TELESCOP", "Teleskop"),
    ("PLATEID",  "Ploca (plate ID)"),
    ("MJD",      "Datum posmatranja (MJD)"),
    ("FIBERID",  "Broj vlakna (fiber ID)"),
    ("RA",       "Rektascenzija [deg]"),
    ("DEC",      "Deklinacija [deg]"),
    ("EXPTIME",  "Ukupno vreme ekspozicije [s]"),
    ("QUALITY",  "Kvalitet posmatranja"),
]

with fits.open(putanja_fits) as hdul:
    h0 = hdul[0].header

    meta = {}
    for kljuc, opis in VAZNI_KLJUCEVI:
        if kljuc in h0:
            meta[opis] = h0[kljuc]

    so = hdul[2].data
    meta["Klasa objekta"] = so["CLASS"][0].strip()
    meta["Podklasa"]      = so["SUBCLASS"][0].strip()
    meta["Crveni pomak z"]= f"{so['Z'][0]:.6f}"
    meta["Greska z"]      = f"{so['Z_ERR'][0]:.6f}"
    meta["S/N (medijana)"]= f"{so['SN_MEDIAN_ALL'][0]:.2f}"

    d       = hdul[1].data
    loglam  = d["loglam"].astype(float)
    fluks   = d["flux"].astype(float)
    ivar    = d["ivar"].astype(float)
    and_mask= d["and_mask"]

vazeci   = (and_mask == 0) & (ivar > 0)
talasna  = 10.0 ** loglam
med      = np.median(fluks[vazeci])
rel_fluks= fluks / med * 100.0

print(f"Piksela ukupno: {len(talasna)}, važećih: {int(vazeci.sum())}")
for opis, vrednost in meta.items():
    print(f"  {opis}: {vrednost}")

## 4. Prikaz spektra

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(talasna[vazeci], rel_fluks[vazeci], lw=0.6, color="steelblue")
ax.set_xlabel("Talasna dužina (Å)")
ax.set_ylabel("Relativni fluks")
ax.set_title(
    f"{os.path.basename(putanja_fits)}  "
    f"— {meta.get('Klasa objekta','?')} {meta.get('Podklasa','?')}"
    f"  z = {meta.get('Crveni pomak z','?')}"
)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Čuvanje CSV fajla

In [ ]:
putanja_csv = os.path.splitext(putanja_fits)[0] + ".csv"

with open(putanja_csv, "w", encoding="utf-8") as f:
    # f.write(f"# Izvor: {os.path.basename(putanja_fits)}\n")
    # for opis, vrednost in meta.items():
    #     f.write(f"# {opis}: {vrednost}\n")
    # f.write("# Normalizacija: fluks / medijana(vazecih_piksela) * 100\n")
    # f.write("#\n")
    f.write("Talasna_duzina_A,Relativni_fluks\n")
    for lam, rf, ok in zip(talasna, rel_fluks, vazeci):
        if ok:
            f.write(f"{lam:.2f},{rf:.4f}\n")

print(f"Sačuvano: {putanja_csv}")

## 6. Preuzimanje CSV fajla

In [ ]:
files.download(putanja_csv)